In [1]:
# Stage 5 — PPE Representation Study
# 5.5 — Temporal Existence Matrix (B'')
#
# Goal: Replace binary pollinator existence matrix P with a temporal
# binary existence matrix TMp, where TMp[species, bin, week] = 1 if
# any GBIF observation exists in that (species, bin, week), else 0.
# No MIN_OBS filter — any single observation counts as present.
# PCA TMp to 15D → Vp_temp. Pair with existing binary Vf.
# Feature vector: [Vf (15D), Vp_temp (15D), N (1D)] = 31D
# Compare against A2, A3, and B' (PMp).

import numpy as np
import pandas as pd
import pickle
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

BASE = "/scratch/ariana.l/Stage 4 Link Prediction Model/"
STAGE5_BASE = "/scratch/ariana.l/Stage 5 PPE Representation Study/"
GBIF_INSECT = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007192_observations_v2.csv"
GBIF_BIRD   = "/scratch/ariana.l/Plant Pollinator Initial Analysis/gbif_0007204_observations_v2.csv"

In [2]:
# Cell 2 — Load existence matrices and reconstruct common bins

print("Loading existence matrices...")
F = pd.read_csv(BASE + "stage4_F_existence_phenofield.csv", index_col=0)
P = pd.read_csv(BASE + "stage4_P_existence_gbif_combined.csv", index_col=0)

common_bins = sorted(set(F.columns) & set(P.columns))
bin_to_idx = {b: i for i, b in enumerate(common_bins)}
n_bins = len(common_bins)  # 3160
n_weeks = 52
print(f"Common bins: {n_bins}")

def snap(x):
    return round(round(x * 2) / 2, 1)

def fmt(x):
    return f"{x:.1f}"

Loading existence matrices...
Common bins: 3160


In [3]:
# Cell 3 — Load and combine GBIF observations (optimized)

def process_gbif(path, label):
    print(f"Loading {label}...")
    df = pd.read_csv(path, usecols=['pollinator_species', 'lat', 'lon', 'doy'],
                     low_memory=False)
    print(f"  {label}: {len(df):,} rows")
    
    # Convert doy to week index
    df['week'] = ((df['doy'] - 1) // 7).clip(0, 51).astype(np.int8)
    
    # Snap and filter to common bins immediately — before concat
    df['bin_key'] = df['lat'].map(snap).map(fmt) + '_' + df['lon'].map(snap).map(fmt)
    df = df[df['bin_key'].isin(bin_to_idx)]
    df['bin_idx'] = df['bin_key'].map(bin_to_idx).astype(np.int16)
    
    # Drop columns we no longer need
    df = df[['pollinator_species', 'bin_idx', 'week']].dropna()
    print(f"  {label} after filtering: {len(df):,} rows")
    return df

ins  = process_gbif(GBIF_INSECT, "insects")
bird = process_gbif(GBIF_BIRD, "birds")

gbif = pd.concat([ins, bird], ignore_index=True)
print(f"\nCombined after filtering: {len(gbif):,} rows")

Loading insects...
  insects: 5,053,625 rows
  insects after filtering: 4,519,114 rows
Loading birds...
  birds: 542,898,437 rows
  birds after filtering: 499,828,068 rows

Combined after filtering: 504,347,182 rows


In [4]:
# Cell 4 — Build TMp (n_species × n_bins × n_weeks), binary

# Get unique (species, bin, week) combinations — no normalization needed
print("Finding unique (species, bin, week) combinations...")
presence = gbif.groupby(['pollinator_species', 'bin_idx', 'week']).size().reset_index()[
    ['pollinator_species', 'bin_idx', 'week']
]
print(f"Unique presence records: {len(presence):,}")

# Ordered pollinator species list
pol_species = sorted(gbif['pollinator_species'].dropna().unique())
pol_to_idx = {s: i for i, s in enumerate(pol_species)}
n_species = len(pol_species)
print(f"Pollinator species: {n_species}")

# Allocate TMp — binary, so uint8 to save memory
print(f"Allocating TMp: ({n_species}, {n_bins}, {n_weeks})...")
TMp = np.zeros((n_species, n_bins, n_weeks), dtype=np.uint8)
print(f"Memory usage: {TMp.nbytes / 1e9:.2f} GB")

# Fill TMp — any observation = 1
print("Filling TMp...")
for row in presence.itertuples(index=False):
    sp_i = pol_to_idx.get(row.pollinator_species)
    if sp_i is not None:
        TMp[sp_i, row.bin_idx, row.week] = 1

print(f"TMp shape: {TMp.shape}")
print(f"Non-zero entries: {np.count_nonzero(TMp):,} / {TMp.size:,} ({100*np.count_nonzero(TMp)/TMp.size:.1f}%)")

Finding unique (species, bin, week) combinations...
Unique presence records: 8,060,041
Pollinator species: 4425
Allocating TMp: (4425, 3160, 52)...
Memory usage: 0.73 GB
Filling TMp...
TMp shape: (4425, 3160, 52)
Non-zero entries: 8,060,041 / 727,116,000 (1.1%)


In [5]:
# Cell 5 — Fit PCA to 15D → Vp_temp

# Cast to float32 for PCA
TMp_flat = TMp.reshape(n_species, n_bins * n_weeks).astype(np.float32)
print(f"Flattened shape: {TMp_flat.shape}")

print("Fitting PCA (randomized, 15 components)...")
pca_tmp = PCA(n_components=15, svd_solver='randomized', random_state=42)
Vp_temp = pca_tmp.fit_transform(TMp_flat)
print(f"Vp_temp shape: {Vp_temp.shape}")
print(f"Variance explained per component: {pca_tmp.explained_variance_ratio_.round(3)}")
print(f"Total variance explained: {pca_tmp.explained_variance_ratio_.sum():.3f}")

Flattened shape: (4425, 164320)
Fitting PCA (randomized, 15 components)...
Vp_temp shape: (4425, 15)
Variance explained per component: [0.423 0.097 0.058 0.031 0.023 0.016 0.013 0.012 0.01  0.008 0.007 0.007
 0.006 0.005 0.005]
Total variance explained: 0.722


In [6]:
# Cell 6 — Save Vp_temp and free memory

Vp_temp_df = pd.DataFrame(Vp_temp, index=pol_species, columns=[f'PC{i+1}' for i in range(15)])
Vp_temp_df.to_csv(STAGE5_BASE + "stage5_Vp_temp.csv")
print(f"Saved Vp_temp: {Vp_temp_df.shape}")

with open(STAGE5_BASE + "stage5_pca_tmp.pkl", "wb") as f:
    pickle.dump(pca_tmp, f)
print("Saved PCA object")

del TMp, TMp_flat
import gc; gc.collect()
print("Freed TMp from memory")

Saved Vp_temp: (4425, 15)
Saved PCA object
Freed TMp from memory


In [7]:
# Cell 7 — Reconstruct training pairs and assemble 31D feature vectors

print("Loading assets...")
Vf_df = pd.read_csv(BASE + "stage4_Vf_phenofield.csv", index_col=0)
globi = pd.read_csv(BASE + "stage4_globi_conus_broad.csv")

F_common = F[common_bins]
P_common = P[common_bins]

# Positive pairs
pos_pairs = globi[['sourceTaxonName', 'targetTaxonName']].drop_duplicates()
pos_pairs.columns = ['pollinator', 'plant']
pos_pairs = pos_pairs[
    pos_pairs['plant'].isin(Vf_df.index) &
    pos_pairs['plant'].isin(F.index) &
    pos_pairs['pollinator'].isin(Vp_temp_df.index) &
    pos_pairs['pollinator'].isin(P.index)
]
pos_pairs['label'] = 1
print(f"Positive pairs: {len(pos_pairs)}")

# Negative pairs
pos_set = set(zip(pos_pairs['pollinator'], pos_pairs['plant']))
all_plants = list(set(Vf_df.index) & set(F.index))
all_pols = list(set(Vp_temp_df.index) & set(P.index))

np.random.seed(42)
neg_pairs = []
while len(neg_pairs) < len(pos_pairs) * 3:
    pol = np.random.choice(all_pols)
    plant = np.random.choice(all_plants)
    if (pol, plant) not in pos_set:
        neg_pairs.append((pol, plant, 0))

neg_pairs = pd.DataFrame(neg_pairs, columns=['pollinator', 'plant', 'label'])
print(f"Negative pairs: {len(neg_pairs)}")

pairs = pd.concat([pos_pairs, neg_pairs], ignore_index=True)

def build_features(row):
    vf = Vf_df.loc[row.plant].values              # 15D — binary plant embedding
    vp = Vp_temp_df.loc[row.pollinator].values    # 15D — temporal pollinator embedding
    N  = float(np.dot(F_common.loc[row.plant].values, P_common.loc[row.pollinator].values))
    return np.concatenate([vf, vp, [N]])           # 31D

print("Assembling feature matrix...")
X = np.vstack([build_features(row) for row in pairs.itertuples()])
y = pairs['label'].values
print(f"X shape: {X.shape}, positive rate: {y.mean():.3f}")

Loading assets...
Positive pairs: 3148
Negative pairs: 9444
Assembling feature matrix...
X shape: (12592, 31), positive rate: 0.250


In [8]:
# Cell 8 — Train/test split and logistic regression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

clf_bdprime = LogisticRegression(max_iter=1000, random_state=42)
clf_bdprime.fit(X_train, y_train)

y_prob = clf_bdprime.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)

print(f"\nB'' (31D, TMp binary) Results:")
print(f"  ROC-AUC: {roc:.3f}")
print(f"  PR-AUC:  {pr:.3f}")
print(f"\nFor reference:")
print(f"  B' (31D, PMp normalized): ROC-AUC 0.924, PR-AUC 0.828")
print(f"  A2 (31D, binary):         ROC-AUC 0.931, PR-AUC 0.842")
print(f"  A3 (32D, scalar delta):   ROC-AUC 0.950, PR-AUC 0.868")

Train: (10073, 31), Test: (2519, 31)

B'' (31D, TMp binary) Results:
  ROC-AUC: 0.925
  PR-AUC:  0.828

For reference:
  B' (31D, PMp normalized): ROC-AUC 0.924, PR-AUC 0.828
  A2 (31D, binary):         ROC-AUC 0.931, PR-AUC 0.842
  A3 (32D, scalar delta):   ROC-AUC 0.950, PR-AUC 0.868
